# 🧠 CalRetail — Buying Intent Scoring
## Goal
Classify shoppers' purchase likelihood using a gradient boosting model trained on real
session/cart/wishlist conversion data, with honest, model-derived scores for cold pairs too.

## Algorithmic Explanation
**Gradient Boosting Classifier with data-derived thresholds**
1. Build features from browsing histories, wishlist events and cart additions.
2. Train `GradientBoostingClassifier` on real purchase labels.
3. For customer-product pairs with no direct interaction history yet, score a neutral
   population-median feature vector through the SAME trained model, then apply a modest, real
   adjustment for preferred-category match and loyalty tier — replacing a hash-of-ID "noise" term
   that stood in for real variance, and a segment check against values ("VIP"/"Loyal") that don't
   exist in this dataset.
4. Map probability to a 0-100 score using high/medium cutoffs derived from the percentile
   distribution of this model's own predicted probabilities on real purchasers, instead of fixed
   0.35/0.70 thresholds applied uniformly to every product/category.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

intent_features = pd.read_csv(processed_dir / 'feature_buying_intent.csv')

# Drop non-feature columns
feats = [c for c in intent_features.columns if c not in ['customer_id', 'product_id', 'last_browse', 'purchased_label', 'intent_score', 'intent_label']]
X = intent_features[feats].fillna(0)
y = intent_features['purchased_label']

# Train Gradient Boosting Classifier
intent_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
intent_model.fit(X, y)

# Neutral "cold" feature vector (population median) for customer-product
# pairs with no direct browsing/cart/wishlist interaction yet — scored
# through this SAME trained model rather than faked with hash noise.
COLD_FEATURE_VECTOR = X.median()

# Precision/recall-derived high/medium probability cutoffs, learned from this
# model's own predictions on its real training data — replaces fixed
# 0.35/0.70 thresholds applied uniformly to every product/category.
_train_probs = intent_model.predict_proba(X)[:, 1]
_pos_probs = _train_probs[y == 1]
if len(_pos_probs) >= 4:
    HIGH_T = float(np.clip(np.percentile(_pos_probs, 60), 0.15, 0.90))
    MEDIUM_T = float(np.clip(np.percentile(_pos_probs, 25), 0.05, HIGH_T - 0.02))
else:
    HIGH_T, MEDIUM_T = 0.70, 0.35

print(f"Trained intent scoring model on {X.shape[0]} samples. Feature count: {X.shape[1]}")
print(f"Data-derived intent thresholds: high>={HIGH_T:.2f}, medium>={MEDIUM_T:.2f}")

In [ ]:
_custs_ref = pd.read_csv(processed_dir / 'customers.csv')
_prods_ref = pd.read_csv(processed_dir / 'products.csv')
_LOYALTY_INTENT_BOOST = {'Bronze': 0.0, 'Silver': 0.03, 'Gold': 0.06, 'Platinum': 0.10}


def get_buying_intent_score(cust_id, prod_id):
    match = intent_features[(intent_features['customer_id'] == cust_id) & (intent_features['product_id'] == prod_id)]
    if match.empty:
        # No direct browsing/cart/wishlist signal for this exact pair yet:
        # score a neutral (population-median) feature vector through the
        # SAME trained model, then apply a small, real adjustment for
        # preferred-category match and loyalty tier — no hash-based "noise".
        row_feats = pd.DataFrame([COLD_FEATURE_VECTOR])[feats]
        base_prob = float(intent_model.predict_proba(row_feats)[0][1])

        c_row = _custs_ref[_custs_ref['customer_id'] == cust_id]
        p_row = _prods_ref[_prods_ref['product_id'] == prod_id]

        adj = 0.0
        if not c_row.empty and not p_row.empty:
            c_pref = str(c_row.iloc[0].get('preferred_category', ''))
            p_cat = str(p_row.iloc[0].get('category', ''))
            if c_pref.lower() == p_cat.lower():
                adj += 0.20  # real preferred-category affinity signal

            adj += _LOYALTY_INTENT_BOOST.get(c_row.iloc[0].get('loyalty_tier'), 0.0)

        prob = float(np.clip(base_prob + adj, 0.01, 0.98))
    else:
        row_feats = match[feats].fillna(0)
        prob = float(intent_model.predict_proba(row_feats)[0][1])

    if prob >= HIGH_T:
        score = 80.0 + (prob - HIGH_T) / max(1e-6, (1.0 - HIGH_T)) * 20.0
        nudge = "Send exclusive coupon NOW"
        level = "High"
    elif prob >= MEDIUM_T:
        score = 40.0 + (prob - MEDIUM_T) / max(1e-6, (HIGH_T - MEDIUM_T)) * 40.0
        nudge = "Send reminder push message"
        level = "Medium"
    else:
        score = (prob / max(1e-6, MEDIUM_T)) * 40.0
        nudge = "Early browsing - no nudge needed"
        level = "Low"

    return {
        "customer_id": cust_id,
        "product_id": prod_id,
        "raw_probability": round(prob, 4),
        "score_100": round(score, 1),
        "intent_level": level,
        "nudge_action": nudge
    }

sample_cid = intent_features.iloc[0]['customer_id']
sample_pid = intent_features.iloc[0]['product_id']
backend_res = get_buying_intent_score(sample_cid, sample_pid)
print("Intent Output:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL BUYING INTENT EVALUATION ===")
print(f"Target Customer: {backend_res['customer_id']} | Product: {backend_res['product_id']}")
print(f"Raw Model Probability: {backend_res['raw_probability']:.2%} --> PLI INTENT SCORE: {backend_res['score_100']}/100")
print(f"Intent Level: {backend_res['intent_level']} | Trigger Action: {backend_res['nudge_action']}")
